# LIBERO 20Hz CSV解析（1/3/5/10 steps）

このノートブックは `/work/csv/libero/20Hz` 配下のCSVを整理して解析します。
- 対象CSV: `episode_log.csv`（必要に応じて `task_log.csv` も確認可能）
- `tmp` ディレクトリ配下は除外
- run単位で指標を計算し、condition(step数)ごとに平均値・標準偏差を算出
- 平均値±標準偏差のエラーバーを作成

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

In [ ]:
ROOT_DIR = Path('/work/csv/libero/20Hz')
TARGET_CONDITIONS = ['1steps', '3steps', '5steps (default)', '10steps']

def is_not_tmp(path: Path) -> bool:
    return 'tmp' not in {p.lower() for p in path.parts}

def parse_replan_steps(condition: str) -> float:
    m = re.search(r'(\d+)steps', condition)
    return float(m.group(1)) if m else np.nan

episode_paths = []
task_paths = []

for cond in TARGET_CONDITIONS:
    cond_dir = ROOT_DIR / cond
    all_eps = sorted(cond_dir.glob('**/episode_log.csv'))
    all_task = sorted(cond_dir.glob('**/task_log.csv'))

    eps = [p for p in all_eps if is_not_tmp(p)]
    tsk = [p for p in all_task if is_not_tmp(p)]

    episode_paths.extend(eps)
    task_paths.extend(tsk)

    print(f'[{cond}] episode_log: {len(eps)} / task_log: {len(tsk)}')

print(f'合計 episode_log: {len(episode_paths)}')
print(f'合計 task_log: {len(task_paths)}')

In [ ]:
def meta_from_path(csv_path: Path, root_dir: Path) -> dict:
    # 想定: <condition>/<suite>/<run_id>/episode_log.csv
    rel = csv_path.relative_to(root_dir)
    parts = rel.parts

    condition = parts[0] if len(parts) >= 1 else 'unknown'
    suite = parts[1] if len(parts) >= 2 else 'unknown'
    run_id = parts[2] if len(parts) >= 3 else 'unknown'

    return {
        'condition': condition,
        'suite': suite,
        'run_id': run_id,
        'replan_steps': parse_replan_steps(condition),
        'source_file': str(csv_path),
    }

def load_episode_logs(paths, root_dir: Path) -> pd.DataFrame:
    frames = []
    for p in paths:
        try:
            df = pd.read_csv(p)
            meta = meta_from_path(p, root_dir)
            for k, v in meta.items():
                df[k] = v
            frames.append(df)
        except Exception as e:
            print(f'[警告] 読み込み失敗: {p}: {e}')

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [ ]:
episode_df = load_episode_logs(episode_paths, ROOT_DIR)
print('episode_df shape:', episode_df.shape)
episode_df.head(3)

In [ ]:
# 型の正規化
if not episode_df.empty:
    if 'success' in episode_df.columns:
        episode_df['success'] = (
            episode_df['success']
            .astype(str).str.lower()
            .map({'true': 1, 'false': 0, '1': 1, '0': 0})
            .fillna(0)
            .astype(int)
        )

    for col in ['steps', 'avg_latency_ms', 'replan_steps']:
        if col in episode_df.columns:
            episode_df[col] = pd.to_numeric(episode_df[col], errors='coerce')

In [ ]:
# run単位（condition x suite x run_id）の集計
run_summary = (
    episode_df
    .groupby(['condition', 'replan_steps', 'suite', 'run_id'], as_index=False)
    .agg(
        episodes=('episode_idx', 'count'),
        success_rate=('success', 'mean'),
        steps_mean=('steps', 'mean'),
        latency_mean_ms=('avg_latency_ms', 'mean'),
    )
)

run_summary = run_summary.sort_values(['suite', 'replan_steps', 'run_id'])
run_summary.head(10)

In [ ]:
# suite x step 条件の mean/std（run間のばらつき）
suite_step_stats = (
    run_summary
    .groupby(['suite', 'condition', 'replan_steps'], as_index=False)
    .agg(
        runs=('run_id', 'nunique'),
        success_rate_mean=('success_rate', 'mean'),
        success_rate_std=('success_rate', 'std'),
        steps_mean=('steps_mean', 'mean'),
        steps_std=('steps_mean', 'std'),
        latency_mean_ms=('latency_mean_ms', 'mean'),
        latency_std_ms=('latency_mean_ms', 'std'),
    )
)

# 1runしか無い条件はstdがNaNになるので0埋め
for c in ['success_rate_std', 'steps_std', 'latency_std_ms']:
    suite_step_stats[c] = suite_step_stats[c].fillna(0.0)

suite_step_stats = suite_step_stats.sort_values(['suite', 'replan_steps'])
display(suite_step_stats)

In [ ]:
# 可視化1: suite別 成功率（mean ± std）
if not suite_step_stats.empty:
    suites = sorted(suite_step_stats['suite'].unique())
    fig, ax = plt.subplots(figsize=(10, 5))

    for suite in suites:
        d = suite_step_stats[suite_step_stats['suite'] == suite].sort_values('replan_steps')
        ax.errorbar(
            d['replan_steps'], d['success_rate_mean'], yerr=d['success_rate_std'],
            marker='o', capsize=4, linewidth=1.8, label=suite
        )

    ax.set_title('20Hz: step数ごとの成功率（平均±標準偏差）')
    ax.set_xlabel('replan_steps')
    ax.set_ylabel('success_rate')
    ax.set_ylim(0, 1.0)
    ax.grid(alpha=0.3)
    ax.legend(title='suite', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# 可視化2: suite別 平均レイテンシ（mean ± std）
if not suite_step_stats.empty:
    suites = sorted(suite_step_stats['suite'].unique())
    fig, ax = plt.subplots(figsize=(10, 5))

    for suite in suites:
        d = suite_step_stats[suite_step_stats['suite'] == suite].sort_values('replan_steps')
        ax.errorbar(
            d['replan_steps'], d['latency_mean_ms'], yerr=d['latency_std_ms'],
            marker='o', capsize=4, linewidth=1.8, label=suite
        )

    ax.set_title('20Hz: step数ごとの平均レイテンシ（平均±標準偏差）')
    ax.set_xlabel('replan_steps')
    ax.set_ylabel('latency_mean_ms')
    ax.grid(alpha=0.3)
    ax.legend(title='suite', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# エクスポート
out_dir = Path('/work/csv/libero/analysis_exports')
out_dir.mkdir(parents=True, exist_ok=True)

suite_file = out_dir / 'suite_step_stats_20Hz_mean_std.csv'
run_file = out_dir / 'run_summary_20Hz.csv'

suite_step_stats.to_csv(suite_file, index=False)
run_summary.to_csv(run_file, index=False)

print(f'書き出し完了: {suite_file}')
print(f'書き出し完了: {run_file}')